# tiny-infer: T4 runbook (every number in the README comes from this notebook)

Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. Total ≈ 75-100 min, almost all of it benchmarks.
Order matters: vLLM is installed **last** because it replaces the PyTorch build.

| Step | Phase | Output |
|---|---|---|
| 1-3 | setup, KV math, unit tests | sanity |
| 4 | kernel build + self-test | `torch.ops.tiny_infer.*` vs PyTorch reference |
| 5 | correctness oracle on TinyLlama | token-identical to HF |
| 6 | HF baseline (Phase 1) | `results/hf_baseline.json` |
| 7-9 | naive server / engine / engine+kernels (Phases 2, 3, 5) | `results/serving.jsonl` |
| 10 | paged vs contiguous (Phase 4) | `results/kv_capacity.json` |
| 11 | kernel op + end-to-end deltas (Phase 5) | `results/kernel_bench.json` |
| 12 | prefix caching, chunked prefill (differentiators) | `results/serving.jsonl` |
| 13 | report | `results/REPORT.md`, `results/serving.png` |
| 14 | vLLM (Phase 6) + final report | |

In [ ]:
# 1. GPU + repo
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
if not os.path.exists("tiny-infer"):
    !git clone -q https://github.com/Musharaf-khan/tiny-infer.git
%cd tiny-infer
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())

In [ ]:
# 2. KV-cache math (the interview staple)
!python scripts/kv_math.py --context 2048 --users 32

In [ ]:
# 3. Unit tests (CPU, tiny random Llama vs HF: scheduler, block manager, prefix cache, preemption, HTTP/SSE)
!python -m pytest tests -q

In [ ]:
# 4. Build the Project 1 kernels (first build ~1-2 min) and check them against the PyTorch reference
!python -m tiny_infer.kernels

In [ ]:
# 5. Correctness oracle on the real model: greedy output must match HF token for token
!python scripts/check_correctness.py --dtype float32
!python scripts/check_correctness.py --dtype float16
!python scripts/check_correctness.py --dtype float16 --custom-kernels

In [ ]:
# 6. Phase 1 baseline: HF generate() in-process (concurrency 1 + static batching)
!python bench/hf_baseline.py --batch-sizes 1 4 16 32

In [ ]:
# 7. Phase 2: naive server (one generate() at a time). Watch TTFT explode as concurrency rises.
!python scripts/run_serving_bench.py --label naive --server "python -m tiny_infer.server --backend naive" -- --concurrency 1 4 16 --num-prompts 16

In [ ]:
# 8. Phases 3+4: tiny-infer, continuous batching + paged KV cache (PyTorch ops)
!python scripts/run_serving_bench.py --label engine --server "python -m tiny_infer.server --max-num-seqs 128" -- --concurrency 1 4 16 32 64

In [ ]:
# 9. Phase 5: same engine with the Project 1 kernels in the decode path
!python scripts/run_serving_bench.py --label engine+kernels --server "python -m tiny_infer.server --max-num-seqs 128 --custom-kernels" -- --concurrency 1 4 16 32 64

In [ ]:
# 10. Phase 4: paged vs contiguous on a fixed 1 GiB KV pool (memory is the binding constraint)
!python bench/kv_capacity.py --kv-cache-gb 1.0 --num-prompts 256

In [ ]:
# 11. Phase 5: op-level and end-to-end kernel deltas
!python bench/kernel_bench.py --e2e-batches 1 16 32 64

In [ ]:
# 11b. Where a decode step's time goes (evidence for the vLLM gap analysis)
!python bench/step_profile.py --batch 32
!python bench/step_profile.py --batch 32 --custom-kernels

In [ ]:
# 12a. Prefix caching: every prompt shares a 1024-token system prompt
!python scripts/run_serving_bench.py --label prefix-off --server "python -m tiny_infer.server" -- --concurrency 16 --num-prompts 64 --input-len 1100 1300 --output-len 32 64 --shared-prefix 1024
!python scripts/run_serving_bench.py --label prefix-on --server "python -m tiny_infer.server --prefix-caching" -- --concurrency 16 --num-prompts 64 --input-len 1100 1300 --output-len 32 64 --shared-prefix 1024
# 12b. Chunked prefill: long and short prompts mixed; look at p99 ITL (decode stalls behind a long prefill)
!python scripts/run_serving_bench.py --label nochunk --server "python -m tiny_infer.server --no-chunked-prefill" -- --concurrency 16 --num-prompts 64 --input-len 64 1900 --output-len 64 128
!python scripts/run_serving_bench.py --label chunk-512 --server "python -m tiny_infer.server --max-num-batched-tokens 512" -- --concurrency 16 --num-prompts 64 --input-len 64 1900 --output-len 64 128

In [ ]:
# 13. Report (tiny-infer numbers). Download results/ now in case the vLLM install breaks the runtime.
!python bench/make_report.py
from IPython.display import Image, Markdown, display
display(Image("results/serving.png"))
!zip -qr results_tiny_infer.zip results

## 14. vLLM comparison (run last: installing vLLM replaces PyTorch)
Same harness, same workload, same concurrency levels. `--dtype half` because the T4 has no bf16;
prefix caching off because tiny-infer runs without it in the main table.

In [ ]:
!pip install -q vllm
!python -c "import vllm; print('vllm', vllm.__version__)" | tee results/vllm_version.txt
!python scripts/run_serving_bench.py --label vllm --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 --startup-timeout 1200 --server "vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0 --dtype half --max-model-len 2048 --max-num-seqs 128 --gpu-memory-utilization 0.90 --no-enable-prefix-caching --port 8000" -- --concurrency 1 4 16 32 64
!python bench/make_report.py
!zip -qr results_all.zip results
from google.colab import files; files.download("results_all.zip")